# Aplicar la calibración INEN (~63 PJ en 2023) a otro input base

Esta libreta toma cualquier CSV con el mismo esquema que `sisepuede_raw_inputs_*` y le aplica **solo** la modificación necesaria para que la demanda industrial INEN total caiga a ~63 PJ en 2023 y ~99 PJ en 2050 (BAU), replicando la gráfica del NDC de Libia.

## Lógica

La demanda INEN se calcula como

$$\text{demand}_{c,t} = \bigl(\text{prod}_{c,t}\cdot I^{prod}_{c,0} + \text{gdp}_t\cdot I^{gdp}_{c,0}\bigr)\;\times\; \text{scalar}_{c,t}$$

Se escalan las **intensidades iniciales** $I^{prod}_{c,0}$ / $I^{gdp}_{c,0}$ (columnas `consumpinit_inen_energy_*`) y **no** el scalar, porque la transformación `TX:INEN:INC_EFFICIENCY_PRODUCTION_*` sobrescribe `scalar_inen_energy_demand_*` al final del horizonte via `magnitude_type=final_value` — cualquier ajuste al scalar se pierde bajo las estrategias Unconditional/Conditional.

## Parámetros a ajustar

Modifica los valores de la siguiente celda para apuntar a otro input base:

In [ ]:
# Ruta al CSV base (otro `sisepuede_raw_inputs_*.csv`) que queremos recalibrar.
# Por defecto apunta al mismo input ya calibrado; cámbialo para recalibrar otro.
BASE_INPUT_CSV = "../input_data/sisepuede_raw_inputs_LBY_apr_with_gfr.csv"

# Ruta de salida donde se escribirá el CSV recalibrado.
OUT_INPUT_CSV  = "../input_data/sisepuede_raw_inputs_recalibrated.csv"

# Objetivo de demanda INEN a 2023 (PJ); rango aceptable 60–65 PJ
TARGET_2023_PJ = 63.0

# Año inicial (t=0) del input; en los inputs de Libia es 2015
BASE_YEAR = 2015

# Año donde medimos el objetivo (para Libia 2023 = time_period 8)
TARGET_YEAR = 2023

# Si True, corre SISEPUEDE al final para verificar que el ajuste dio en el blanco
RUN_VERIFICATION = True

# Región del input
REGION = "libya"

In [ ]:
import os, sys, pathlib, warnings, logging
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

HERE = pathlib.Path(os.getcwd())
sys.path.insert(0, str(HERE))
from utils.logger_utils import setup_clean_logger, mute_external_loggers
logger = setup_clean_logger('recalib', logging.INFO)
mute_external_loggers(['sisepuede'])

## 1. Medir la demanda INEN actual del input base

Corre **solo el modelo INEN** (sin NemoMod) sobre el input base para conocer la demanda total actual a 2023. El factor de escala necesario será `TARGET_2023_PJ / demanda_actual_2023`.

In [ ]:
import sisepuede.core.attribute_table as att
import sisepuede.manager.sisepuede_examples as sxl
import sisepuede.manager.sisepuede_file_structure as sfs
import sisepuede.manager.sisepuede_models as sm
from ssp_transformations_handler.GeneralUtils import GeneralUtils

_EXAMPLES = sxl.SISEPUEDEExamples()

# Chart categories (same ones shown in the Libya NDC stack plot)
CHART_CATS = [
    'cement', 'chemicals', 'electronics', 'glass', 'lime_and_carbonite',
    'metals', 'mining', 'other_product_manufacturing', 'paper', 'plastic',
    'textiles', 'wood',
]

def run_inen_only(csv_path: str, y0: int = BASE_YEAR, y1: int = 2050) -> pd.DataFrame:
    """Run only the EnergyConsumption (INEN) model, no NemoMod."""
    df_raw = pd.read_csv(csv_path)
    fs = sfs.SISEPUEDEFileStructure(initialize_directories=False)
    key_tp, key_year = fs.model_attributes.dim_time_period, fs.model_attributes.field_dim_year
    years = np.arange(y0, y1 + 1).astype(int)
    att_tp = att.AttributeTable(
        pd.DataFrame({key_tp: range(len(years)), key_year: years}), key_tp,
    )
    fs.model_attributes.update_dimensional_attribute_table(att_tp)
    matt = fs.model_attributes
    models = sm.SISEPUEDEModels(
        matt, allow_electricity_run=False,
        fp_julia=fs.dir_jl, fp_nemomod_reference_files=fs.dir_ref_nemo,
        initialize_julia=False,
    )
    df_ex = _EXAMPLES('input_data_frame')
    df = GeneralUtils().add_missing_cols(df_ex, df_raw.copy())
    df['region'] = REGION
    df = df[df['year'] <= y1]
    df_out = models.project(df, include_electricity_in_energy=False)
    df_out['year'] = df_out['time_period'].apply(lambda t: y0 + int(t))
    return df_out

df_out = run_inen_only(BASE_INPUT_CSV)
chart_cols = [f'energy_demand_inen_{c}' for c in CHART_CATS]
current_2023 = df_out.loc[df_out['year'] == TARGET_YEAR, chart_cols].sum(axis=1).iloc[0]
current_2050 = df_out.loc[df_out['year'] == 2050, chart_cols].sum(axis=1).iloc[0]
print(f'Demanda INEN actual del input base:')
print(f'  {TARGET_YEAR}: {current_2023:.2f} PJ')
print(f'  2050:         {current_2050:.2f} PJ')

## 2. Calcular factor de escala y aplicarlo

El factor se aplica a las 21 columnas de intensidad inicial:

- `consumpinit_inen_energy_tj_per_tonne_production_*` (19 categorías industriales)
- `consumpinit_inen_energy_tj_per_mmm_gdp_other_product_manufacturing`
- `consumpinit_inen_energy_total_pj_agriculture_and_livestock`

Ninguna otra columna se modifica.

In [ ]:
factor = TARGET_2023_PJ / current_2023
print(f'Factor de escala: {factor:.4f}')

df_in = pd.read_csv(BASE_INPUT_CSV)
intensity_cols = [
    c for c in df_in.columns
    if c.startswith('consumpinit_inen_energy_tj_per_tonne_production_')
    or c.startswith('consumpinit_inen_energy_tj_per_mmm_gdp_')
    or c.startswith('consumpinit_inen_energy_total_pj_')
]
assert len(intensity_cols) > 0, 'No se encontraron columnas consumpinit_inen_energy_*'
print(f'Aplicando factor a {len(intensity_cols)} columnas consumpinit_inen_energy_*')

for c in intensity_cols:
    df_in[c] = df_in[c] * factor

os.makedirs(os.path.dirname(OUT_INPUT_CSV), exist_ok=True)
df_in.to_csv(OUT_INPUT_CSV, index=False)
print(f'Guardado input recalibrado en: {OUT_INPUT_CSV}')

## 3. Verificar la nueva demanda INEN

Corre de nuevo el modelo INEN sobre el input recalibrado para confirmar que 2023 cae en el objetivo.

In [ ]:
df_out_new = run_inen_only(OUT_INPUT_CSV)
new_2023 = df_out_new.loc[df_out_new['year'] == TARGET_YEAR, chart_cols].sum(axis=1).iloc[0]
new_2050 = df_out_new.loc[df_out_new['year'] == 2050, chart_cols].sum(axis=1).iloc[0]
print(f'Demanda INEN recalibrada:')
print(f'  {TARGET_YEAR}: {new_2023:.2f} PJ (objetivo {TARGET_2023_PJ})')
print(f'  2050:         {new_2050:.2f} PJ')
assert abs(new_2023 - TARGET_2023_PJ) < 0.5, 'El objetivo 2023 no se alcanzó dentro de ±0.5 PJ'
print('\nOK — ajuste verificado.')

## 4. (Opcional) Corrida completa con las 3 estrategias

Si `RUN_VERIFICATION=True`, corre SISEPUEDE con BASE / BAU / Unconditional / Conditional sobre el nuevo input y muestra la tabla de demanda INEN por escenario y año. Esto ejecuta el pipeline completo (sin NemoMod por velocidad; pasa `--electricity` al runner si tienes Julia instalado).

In [ ]:
if RUN_VERIFICATION:
    import subprocess, shlex
    runner = HERE / 'run_energy_validation.py'
    cmd = f'python {runner} --strategies 0,6003,6004,6005 --input_csv {os.path.basename(OUT_INPUT_CSV)}'
    print('Ejecutando:', cmd)
    res = subprocess.run(shlex.split(cmd), capture_output=True, text=True, cwd=str(HERE))
    tail = '\n'.join((res.stdout + res.stderr).splitlines()[-30:])
    print(tail)

## Notas

- Si el input base tiene **diferente año inicial** que 2015, ajusta `BASE_YEAR` en la celda de parámetros.
- El factor resultante es proporcional — si el motor de crecimiento (producción IPPU, GDP) del nuevo input crece a una tasa distinta, la trayectoria 2050 será proporcionalmente distinta. El único objetivo garantizado es el valor 2023.
- Este ajuste **solo** modifica 21 columnas `consumpinit_inen_energy_*`. Toda otra columna (fracciones de combustible, eficiencias, scalars, datos de IPPU, SCOE, etc.) queda intacta.
- Para reproducir la gráfica del NDC después de recalibrar, usa `reproduce_inen_energy_chart.ipynb` apuntando al nuevo input.